# 🚀 OpositaIA Fine-Tuning: Mistral 7B + Unsloth (Free Colab)
## Edición: Resume & Save (Entrenamiento Interrumpible)

Este notebook está optimizado para **Google Drive**, permitiendo:
1. Guardar checkpoints cada X pasos.
2. **Reanudar** el entrenamiento si Colab se desconecta.
3. Exportar el modelo final a GGUF para VPS (8GB).

**Pasos Previos**:
1. Montar Google Drive.
2. Subir `golden_dataset/final_v1_train.jsonl` a tu Drive.

In [ ]:
# 0. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

output_dir = "/content/drive/MyDrive/OpositaIA_Models/Mistral_Checkpoint"

In [ ]:
%%capture
# 1. Instalar Unsloth y dependencias
import torch
major_version, minor_version = torch.cuda.get_device_capability()

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 2. Configurar Modelo
from unsloth import FastLanguageModel

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
# 3. Cargar Dataset (Desde Drive)
from datasets import load_dataset

dataset_path = "/content/drive/MyDrive/golden_dataset/final_v1_train.jsonl"
# Asegúrate de subir el archivo a esa ruta o cámbiala aquí

alpaca_prompt = """{instruction}\n\n{input}\n\n{output}""" # Formato simplificado

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for i, inp, out in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction=i, input=inp, output=out) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts, }

dataset = load_dataset("json", data_files = dataset_path, split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# 4. Entrenar con SOPORTE PARA REANUDAR (Checkpoints)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 500, # Pasos totales deseados
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        output_dir = output_dir, # Guardar en Drive
        save_strategy = "steps",
        save_steps = 50, # Guardar checkpoint cada 50 pasos
        save_total_limit = 2, # Mantener solo los 2 últimos para ahorrar espacio
    ),
)

# INTENTAR REANUDAR SI EXISTE CHECKPOINT
try:
    trainer.train(resume_from_checkpoint=True)
except ValueError:
    print("No se encontró checkpoint válido, iniciando desde cero.")
    trainer.train()

In [ ]:
# 5. Exportar a GGUF (Optimizado para 8GB RAM)
# Cuantización q4_k_m es el balance perfecto velocidad/calidad
model.save_pretrained_gguf("opositaia_mistral_v1_gguf", tokenizer, quantization_method = "q4_k_m")

# Mover a Drive para que no se pierda
!cp opositaia_mistral_v1_gguf-unsloth.gguf /content/drive/MyDrive/OpositaIA_Models/mistral_7b_v0.3_opositaia.gguf